<a href="https://colab.research.google.com/github/ibmm-unibe-ch/H3BERTa/blob/main/H3BERTa_BLOSUM_SCORE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# BLOSUM62

In [ ]:
from Bio.Align import substitution_matrices
import torch

def calculate_blosum_score(amino1, amino2, blosum_matrix,scores):
    #print('BLOSUM COUPLE', amino1,amino2)
    try:
        scores.append(blosum_matrix[(amino1, amino2)])
    except KeyError:
        # if the couple of aa it is not present in the matrix
        scores.append(-5)
    except IndexError:
        # if the couple of aa contains one [UNK]
        scores.append(-5)
    return scores

def calculate_blosum_scores(predicted_sequences, original_sequences):
    '''
    blosum_scores = calculate_blosum_scores(predicted_values, true_values)
    average_blosum_score = sum(blosum_scores) / len(blosum_scores)
    print("Val AVG Blosum for all amino acids:", average_blosum_score)
    '''
    #from Bio.SubsMat import MatrixInfo #in in the older biopython version
    #blosum_matrix = MatrixInfo.blosum62
    blosum_matrix  = substitution_matrices.load('BLOSUM62')
    scores = []
    for pred_seq, orig_seq in zip(predicted_sequences, original_sequences):
        '''per amino acids couple, not per sequence in this case,
        double check if you want to compute it per sequences pair'''
        calculate_blosum_score(pred_seq, orig_seq, blosum_matrix,scores)
    return scores

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
)
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm


import random
import numpy as np
import torch
from transformers import set_seed

SEED = 42  # scegli il tuo numero preferito

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ------------------------
# CONFIG
# ------------------------
MODEL_NAME_OR_PATH = 'Chrode/H3BERTa'
VAL_CSV_PATH = "/content/drive/MyDrive/review/SB1_val.txt"
SEQ_COLUMN_INDEX = 1
BATCH_SIZE = 16
MAX_LENGTH = 100
MLM_PROB = 0.15


# ------------------------
# LOAD MODEL + TOKENIZER
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME_OR_PATH)
model.to(device)
model.eval()

# ------------------------
# LOAD VALIDATION DATA
# ------------------------

print("Loading validation CSV...")
val_df = pd.read_csv(VAL_CSV_PATH, sep=",")

SEQ_COLUMN_NAME = val_df.columns[SEQ_COLUMN_INDEX]
print("Using sequence column:", SEQ_COLUMN_NAME)

val_df = val_df.dropna(subset=[SEQ_COLUMN_NAME])
print(f"Validation samples: {len(val_df)}")

val_dataset = Dataset.from_pandas(val_df)


# ------------------------
# TOKENIZATION
# ------------------------
def tokenize_fn(examples):
    return tokenizer(
        examples[SEQ_COLUMN_NAME],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True
    )

print("Tokenizing validation set...")
val_dataset = val_dataset.map(tokenize_fn, batched=True)

cols_to_keep = ["input_ids", "attention_mask"]
cols_to_remove = [c for c in val_dataset.column_names if c not in cols_to_keep]
val_dataset = val_dataset.remove_columns(cols_to_remove)

print(val_dataset)


# ------------------------
# DATA COLLATOR + DATALOADER
# ------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

g = torch.Generator()
g.manual_seed(SEED)

eval_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,          # più semplice per la riproducibilità
    generator=g,
)

print("Eval dataloader ready.")


# ------------------------
# VALIDATION LOOP
# ------------------------
losses = []
predicted_values = []
true_values = []

print("Running validation...")
for step, batch in enumerate(tqdm(eval_dataloader)):

    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits  # [batch, seq_len, vocab]

    for i in range(batch["input_ids"].size(0)):
        labels = batch["labels"][i]  # [seq_len]

        labels_index = torch.nonzero(labels != -100, as_tuple=False).squeeze(-1)

        if labels_index.numel() == 0:
            continue

        # ----- TRUE TOKENS -----
        true_token_ids = labels[labels_index]              # ids
        true_tokens_str = tokenizer.decode(true_token_ids)
        true_tokens = [t for t in true_tokens_str.split() if t.strip()]
        true_values.extend(true_tokens)

        # ----- PREDICTED TOKENS -----
        # logits[i, labels_index] -> [num_positions, vocab]
        predicted_token_ids = logits[i, labels_index].argmax(dim=-1)
        predicted_tokens_str = tokenizer.decode(predicted_token_ids)
        predicted_tokens = [t for t in predicted_tokens_str.split() if t.strip()]
        predicted_values.extend(predicted_tokens)

        if len(true_tokens) != len(predicted_tokens):
            print('*** CHECK DIFFERENT LENGTH ***')
            print("pred values:", predicted_tokens, "len:", len(predicted_tokens))
            print("true values:", true_tokens, "len:", len(true_tokens))

    # ----- LOSS -----
    loss = outputs.loss
    losses.append(loss.detach().cpu())

# ------------------------
# METRICS
# ------------------------
val_loss = torch.stack(losses).mean().item()
print(f"\nValidation Loss: {val_loss:.4f}")

accuracy = accuracy_score(true_values, predicted_values)
print("Validation Token Accuracy:", accuracy)

blosum_scores = calculate_blosum_scores(predicted_values, true_values)
if len(blosum_scores) > 0:
    average_blosum_score = sum(blosum_scores) / len(blosum_scores)
else:
    average_blosum_score = float("nan")
print("Val AVG BLOSUM:", average_blosum_score)


Using device: cuda
Loading validation CSV...
Using sequence column: AKEGTSLARSSLYGDPFDY
Validation samples: 1800418
Tokenizing validation set...


Map:   0%|          | 0/1800418 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 1800418
})
Eval dataloader ready.
Running validation...


  0%|          | 0/112527 [00:00<?, ?it/s]


Validation Loss: 1.5769
Validation Token Accuracy: 0.5534776674821417
Val AVG BLOSUM: 2.5478021764190406


# BLOSUM50

In [ ]:
from Bio.Align import substitution_matrices
import torch

def calculate_blosum_score(amino1, amino2, blosum_matrix,scores):
    #print('BLOSUM COUPLE', amino1,amino2)
    try:
        scores.append(blosum_matrix[(amino1, amino2)])
    except KeyError:
        # if the couple of aa it is not present in the matrix
        scores.append(-5)
    except IndexError:
        # if the couple of aa contains one [UNK]
        scores.append(-5)
    return scores

def calculate_blosum_scores(predicted_sequences, original_sequences):
    '''
    blosum_scores = calculate_blosum_scores(predicted_values, true_values)
    average_blosum_score = sum(blosum_scores) / len(blosum_scores)
    print("Val AVG Blosum for all amino acids:", average_blosum_score)
    '''
    #from Bio.SubsMat import MatrixInfo #in in the older biopython version
    #blosum_matrix = MatrixInfo.blosum62
    blosum_matrix  = substitution_matrices.load('BLOSUM50')
    scores = []
    for pred_seq, orig_seq in zip(predicted_sequences, original_sequences):
        '''per amino acids couple, not per sequence in this case,
        double check if you want to compute it per sequences pair'''
        calculate_blosum_score(pred_seq, orig_seq, blosum_matrix,scores)
    return scores

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
)
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm


import random
import numpy as np
import torch
from transformers import set_seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ------------------------
# CONFIG
# ------------------------
MODEL_NAME_OR_PATH = 'Chrode/H3BERTa'
VAL_CSV_PATH = "/content/drive/MyDrive/review/SB1_val.txt"
SEQ_COLUMN_INDEX = 1
BATCH_SIZE = 16
MAX_LENGTH = 100
MLM_PROB = 0.15


# ------------------------
# LOAD MODEL + TOKENIZER
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME_OR_PATH)
model.to(device)
model.eval()

# ------------------------
# LOAD VALIDATION DATA
# ------------------------

print("Loading validation CSV...")
val_df = pd.read_csv(VAL_CSV_PATH, sep=",")

SEQ_COLUMN_NAME = val_df.columns[SEQ_COLUMN_INDEX]
print("Using sequence column:", SEQ_COLUMN_NAME)

val_df = val_df.dropna(subset=[SEQ_COLUMN_NAME])
print(f"Validation samples: {len(val_df)}")

val_dataset = Dataset.from_pandas(val_df)


# ------------------------
# TOKENIZATION
# ------------------------
def tokenize_fn(examples):
    return tokenizer(
        examples[SEQ_COLUMN_NAME],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True
    )

print("Tokenizing validation set...")
val_dataset = val_dataset.map(tokenize_fn, batched=True)

cols_to_keep = ["input_ids", "attention_mask"]
cols_to_remove = [c for c in val_dataset.column_names if c not in cols_to_keep]
val_dataset = val_dataset.remove_columns(cols_to_remove)

print(val_dataset)


# ------------------------
# DATA COLLATOR + DATALOADER
# ------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

g = torch.Generator()
g.manual_seed(SEED)

eval_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,          # più semplice per la riproducibilità
    generator=g,
)

print("Eval dataloader ready.")


# ------------------------
# VALIDATION LOOP
# ------------------------
losses = []
predicted_values = []
true_values = []

print("Running validation...")
for step, batch in enumerate(tqdm(eval_dataloader)):

    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits  # [batch, seq_len, vocab]

    for i in range(batch["input_ids"].size(0)):
        labels = batch["labels"][i]  # [seq_len]

        labels_index = torch.nonzero(labels != -100, as_tuple=False).squeeze(-1)

        if labels_index.numel() == 0:
            continue

        # ----- TRUE TOKENS -----
        true_token_ids = labels[labels_index]              # ids
        true_tokens_str = tokenizer.decode(true_token_ids)
        true_tokens = [t for t in true_tokens_str.split() if t.strip()]
        true_values.extend(true_tokens)

        # ----- PREDICTED TOKENS -----
        # logits[i, labels_index] -> [num_positions, vocab]
        predicted_token_ids = logits[i, labels_index].argmax(dim=-1)
        predicted_tokens_str = tokenizer.decode(predicted_token_ids)
        predicted_tokens = [t for t in predicted_tokens_str.split() if t.strip()]
        predicted_values.extend(predicted_tokens)

        if len(true_tokens) != len(predicted_tokens):
            print('*** CHECK DIFFERENT LENGTH ***')
            print("pred values:", predicted_tokens, "len:", len(predicted_tokens))
            print("true values:", true_tokens, "len:", len(true_tokens))

    # ----- LOSS -----
    loss = outputs.loss
    losses.append(loss.detach().cpu())

# ------------------------
# METRICS
# ------------------------
val_loss = torch.stack(losses).mean().item()
print(f"\nValidation Loss: {val_loss:.4f}")

accuracy = accuracy_score(true_values, predicted_values)
print("Validation Token Accuracy:", accuracy)

blosum_scores = calculate_blosum_scores(predicted_values, true_values)
if len(blosum_scores) > 0:
    average_blosum_score = sum(blosum_scores) / len(blosum_scores)
else:
    average_blosum_score = float("nan")
print("Val AVG BLOSUM:", average_blosum_score)


Using device: cuda
Loading validation CSV...
Using sequence column: AKEGTSLARSSLYGDPFDY
Validation samples: 1800418
Tokenizing validation set...


Map:   0%|          | 0/1800418 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 1800418
})
Eval dataloader ready.
Running validation...


  0%|          | 0/112527 [00:00<?, ?it/s]


Validation Loss: 1.5769
Validation Token Accuracy: 0.5534776674821417
Val AVG BLOSUM: 3.355425413563903
